In [30]:
import pandas as pd
from pathlib import Path

DATA = Path("../../data")
df_json = pd.read_json(DATA / "weather.json")
print(df_json.shape)
print(df_json.dtypes)

(21, 6)
time                             str
temperature_2m_max           float64
temperature_2m_min           float64
precipitation_sum            float64
city                             str
ingested_at           datetime64[us]
dtype: object


In [31]:
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")

def check(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Sẽ quét khoảng {job.total_bytes_processed / 1024**2:.2f} MB")

def run(sql, max_mb=100):
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=max_mb * 1024**2)
    return client.query(sql, job_config=cfg).to_dataframe()

In [32]:
sql = """
SELECT city, COUNT(*) AS days, MIN(time) AS first_day, MAX(time) AS last_day
FROM `bq-learning-510104.raw.weather_daily`
GROUP BY city
"""
check(sql)

Sẽ quét khoảng 0.00 MB


In [21]:
df = run(sql, max_mb=100)
df

,status,n
0,Returned,18057
1,Processing,36063
2,Complete,45282
3,Shipped,53996
4,Cancelled,27373


In [24]:
df = pd.read_parquet(DATA / "weather.parquet")
df["time"] = df["time"].dt.date
df["ingested_at"] = pd.Timestamp.now(tz="UTC")
print(df.dtypes)
df.head(3)

time                               object
temperature_2m_max                float64
temperature_2m_min                float64
precipitation_sum                 float64
city                                  str
ingested_at           datetime64[us, UTC]
dtype: object


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,city,ingested_at
0,2026-10-01,34.1,27.0,0.4,Ha Noi,2026-10-01 10:53:30.034276+00:00
1,2026-10-02,32.1,25.9,4.4,Ha Noi,2026-10-01 10:53:30.034276+00:00
2,2026-10-03,32.0,26.1,1.2,Ha Noi,2026-10-01 10:53:30.034276+00:00


In [27]:
ds = bigquery.Dataset(f"{PROJECT}.raw")
ds.location = "US"
client.create_dataset(ds, exists_ok=True)
print("OK")

OK


In [28]:
table_id = f"{PROJECT}.raw.weather_daily"

schema = [
    bigquery.SchemaField("time", "DATE"),
    bigquery.SchemaField("temperature_2m_max", "FLOAT"),
    bigquery.SchemaField("temperature_2m_min", "FLOAT"),
    bigquery.SchemaField("precipitation_sum", "FLOAT"),
    bigquery.SchemaField("city", "STRING"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]

job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition="WRITE_TRUNCATE",
)

job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
job.result()
print("Đã nạp", job.output_rows, "dòng")

d:\Installsoftware\Anaconda\envs\bi\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Đã nạp 21 dòng


In [29]:
t = client.get_table(table_id)
print("Số dòng:", t.num_rows)
for f in t.schema:
    print(f.name, f.field_type)

Số dòng: 21
time DATE
temperature_2m_max FLOAT
temperature_2m_min FLOAT
precipitation_sum FLOAT
city STRING
ingested_at TIMESTAMP


In [33]:
df_check = run(sql)
df_check

,city,days,first_day,last_day
0,Ha Noi,7,2026-10-01,2026-10-07
1,TP HCM,7,2026-10-01,2026-10-07
2,Da Nang,7,2026-10-01,2026-10-07


In [34]:
cfg_append = bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_APPEND")
client.load_table_from_dataframe(df, table_id, job_config=cfg_append).result()
print("Số dòng sau APPEND:", client.get_table(table_id).num_rows)

d:\Installsoftware\Anaconda\envs\bi\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Số dòng sau APPEND: 42


In [35]:
client.load_table_from_dataframe(df, table_id, job_config=job_config).result()
print("Số dòng sau TRUNCATE:", client.get_table(table_id).num_rows)

d:\Installsoftware\Anaconda\envs\bi\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Số dòng sau TRUNCATE: 21


In [36]:
import os
from dotenv import load_dotenv


ModuleNotFoundError: No module named 'dotenv'

In [37]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv(dotenv_path="../../.env")
key = os.getenv("WEATHER_API_KEY")
print("Đã đọc được khóa:", key is not None)
print("Độ dài khóa:", len(key) if key else 0)